In [1]:
from pathlib import Path
import sys
import pickle
from io import BytesIO

sys.path.append(str(Path().resolve().parent))
from src.processing.pde_ple import es, PDE
from sentence_transformers import SentenceTransformer
import torch
from src.api.constants.model import MODEL_NAME


/opt/app-root/src/uc202-ipn-rex/.venv/lib/python3.11/site-packages/elasticsearch/_sync/client/__init__.py:397: SecurityWarning: Connecting to 'https://noeyyalp.noe.edf.fr:29203' using TLS with verify_certs=False is insecure
  _transport = transport_class(
/opt/app-root/src/uc202-ipn-rex/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from src.models.model import Model

In [ ]:
    test_df = pd.read_excel("combined")
    if only_2:
        test_df.loc[test_df["Annotation"] == 1, "Annotation"] = 0
    test_df.loc[test_df["Annotation"] == 2, "Annotation"] = 1

    test_df=test_df[test_df.Annotation==1]

In [ ]:
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
import time

def evaluate_model(
    model_path, index_name, test_path, k_values=[20, 50, 100], only_2=False
):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = SentenceTransformer(model_path, device=device)

    # --- Instruction for queries ---
    task = "Given a query, retrieve relevant passages that answer the query"

    test_df = pd.read_excel(test_path)
    if only_2:
        test_df.loc[test_df["Annotation"] == 1, "Annotation"] = 0
    test_df.loc[test_df["Annotation"] == 2, "Annotation"] = 1

    test_df = test_df[test_df.Annotation == 1]

    q2answers = (
        test_df.groupby("Question")
        .apply(lambda x: list(zip(x["Content"], x["Annotation"])))
        .to_dict()
    )

    hitrate = {k: 0 for k in k_values}
    precision = {k: 0 for k in k_values}
    total_qs = len(q2answers)

    for q, answers in q2answers.items():
        # --- Apply instruction ---
        instructed_query = f"Instruct: {task}\nQuery: {q}"

        with torch.no_grad():
            query_embedding = model.encode(
                instructed_query, convert_to_numpy=True, normalize_embeddings=True
            ).tolist()

        max_k = max(k_values)
        query_body = {
            "size": max_k,
            "query": {
                "script_score": {
                    "query": {"match_all": {}},
                    "script": {
                        "source": "cosineSimilarity(params.query_vector, 'embedding') + 1.0",
                        "params": {"query_vector": query_embedding},
                    },
                }
            },
        }

        response = es.search(index=index_name, body=query_body)
        retrieved_chunks = [
            hit["_source"]["chunk_content"] for hit in response["hits"]["hits"]
        ]

        relevant_answers = set([a for a, label in answers if label == 1])

        for k in k_values:
            topk = set(retrieved_chunks[:k])
            relevant_retrieved_chunks = set(a for a in topk if a in relevant_answers)
            hitrate[k] += len(relevant_retrieved_chunks)
            precision[k] += len(relevant_retrieved_chunks) / k

    hitrate = {k: v / test_df.shape[0] for k, v in hitrate.items()}
    precision = {k: v / total_qs for k, v in precision.items()}

    return {"hit_rate": hitrate, "precision": precision}


In [7]:
model = Model(model_name="intfloat_multilingual_e5_large_instruct")
evaluate_model(
    model_path=model.local_path,
    index_name="uc202-rex-embeddings-e5",
    test_path="/opt/app-root/src/uc202-ipn-rex/notebooks/test_set.xlsx",
    k_values=[3, 5, 10, 20, 30, 50, 100],
    only_2=False,
) 


/tmp/ipykernel_53673/3025941553.py:27: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: list(zip(x["Content"], x["Annotation"])))
/tmp/ipykernel_53673/3025941553.py:58: DeprecationWarning: The 'body' parameter is deprecated and will be removed in a future version. Instead use individual parameters.
  response = es.search(index=index_name, body=query_body)
/opt/app-root/src/uc202-ipn-rex/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'noeyyalp.noe.edf.fr'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnin

{'hit_rate': {3: 0.0,
  5: 0.0,
  10: 0.017543859649122806,
  20: 0.017543859649122806,
  30: 0.017543859649122806,
  50: 0.017543859649122806,
  100: 0.03508771929824561},
 'precision': {3: 0.0,
  5: 0.0,
  10: 0.0029411764705882353,
  20: 0.0014705882352941176,
  30: 0.000980392156862745,
  50: 0.0005882352941176471,
  100: 0.0005882352941176471}}

In [8]:
evaluate_model(
    model_path="/opt/app-root/src/uc202-ipn-rex/notebooks/models/test",
    index_name="uc202-rex-embeddings-e5-trained",
    test_path="/opt/app-root/src/uc202-ipn-rex/notebooks/test_set.xlsx",
    k_values=[3, 5, 10, 20, 30, 50, 100],
    only_2=False,
)


/tmp/ipykernel_53673/3025941553.py:27: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: list(zip(x["Content"], x["Annotation"])))
/tmp/ipykernel_53673/3025941553.py:58: DeprecationWarning: The 'body' parameter is deprecated and will be removed in a future version. Instead use individual parameters.
  response = es.search(index=index_name, body=query_body)
/opt/app-root/src/uc202-ipn-rex/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'noeyyalp.noe.edf.fr'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnin

{'hit_rate': {3: 0.0,
  5: 0.0,
  10: 0.0,
  20: 0.08771929824561403,
  30: 0.08771929824561403,
  50: 0.08771929824561403,
  100: 0.15789473684210525},
 'precision': {3: 0.0,
  5: 0.0,
  10: 0.0,
  20: 0.007352941176470588,
  30: 0.004901960784313725,
  50: 0.0029411764705882353,
  100: 0.0026470588235294116}}

In [23]:
evaluate_model(
    model_path="/opt/app-root/src/uc202-ipn-rex/notebooks/models/test",
    index_name="uc202-rex-embeddings-e5-trained",
    test_path="/opt/app-root/src/uc202-ipn-rex/notebooks/combined_annotations.xlsx",
    k_values=[3, 5, 10, 20, 30, 50, 100],
    only_2=True,
)


/tmp/ipykernel_45291/92907310.py:20: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: list(zip(x["Content"], x["Annotation"])))
/tmp/ipykernel_45291/92907310.py:52: DeprecationWarning: The 'body' parameter is deprecated and will be removed in a future version. Instead use individual parameters.
  response = es.search(index=index_name, body=query_body)
/opt/app-root/src/uc202-ipn-rex/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'noeyyalp.noe.edf.fr'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.w

{'Anomalie thermique combustible': [("C0000149866 DISC DT Lyon - Grande Halle Remontée Fournisseur FA3 - Anomalie sur les données d'interface crayon-combustible Cette anomalie concerne les distributions axiales de puissance considérées dans les études de conception thermomécanique du crayon et de l’assemblage combustible FA3. Les données d’entrée du Rapport De Sûreté (RDS) issues des études de conception thermomécanique sont donc potentiellement impactées par cette anomalie. ['Enjeu sûreté'] ['EPR - FA3'] [] Les distributions axiales de puissance assemblages fournies par DTI à la BU Fuel sont inversées entre le bas et le haut du coeur. Par conséquent, cette inversion conduit à la fourniture de données combustibles erronées (RFP88199RDP) avec impact potentiel sur les études des évènements de type PCC et RRC-A. Données pour calculs MANTA impactées : coefficients de transfert thermique crayon combustible/réfrigérant (UA) Données pour calculs COMBAT potentiellement impactées : conductance 

{3: 0.25, 5: 0.25, 10: 0.5, 20: 0.5, 30: 0.5, 50: 0.75, 100: 0.75}

In [24]:
model = Model(model_name="intfloat_multilingual_e5_large_instruct")
evaluate_model(
    model_path=model.local_path,
    index_name="uc202-rex-embeddings-e5",
    test_path="/opt/app-root/src/uc202-ipn-rex/notebooks/combined_annotations.xlsx",
    k_values=[3, 5, 10, 20, 30, 50, 100],
    only_2=True,
)


/tmp/ipykernel_45291/92907310.py:20: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: list(zip(x["Content"], x["Annotation"])))
/tmp/ipykernel_45291/92907310.py:52: DeprecationWarning: The 'body' parameter is deprecated and will be removed in a future version. Instead use individual parameters.
  response = es.search(index=index_name, body=query_body)
/opt/app-root/src/uc202-ipn-rex/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'noeyyalp.noe.edf.fr'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.w

{'Anomalie thermique combustible': [("C0000149866 DISC DT Lyon - Grande Halle Remontée Fournisseur FA3 - Anomalie sur les données d'interface crayon-combustible Cette anomalie concerne les distributions axiales de puissance considérées dans les études de conception thermomécanique du crayon et de l’assemblage combustible FA3. Les données d’entrée du Rapport De Sûreté (RDS) issues des études de conception thermomécanique sont donc potentiellement impactées par cette anomalie. ['Enjeu sûreté'] ['EPR - FA3'] [] Les distributions axiales de puissance assemblages fournies par DTI à la BU Fuel sont inversées entre le bas et le haut du coeur. Par conséquent, cette inversion conduit à la fourniture de données combustibles erronées (RFP88199RDP) avec impact potentiel sur les études des évènements de type PCC et RRC-A. Données pour calculs MANTA impactées : coefficients de transfert thermique crayon combustible/réfrigérant (UA) Données pour calculs COMBAT potentiellement impactées : conductance 

{3: 0.0, 5: 0.0, 10: 0.0, 20: 0.0, 30: 0.25, 50: 0.5, 100: 0.75}

In [9]:
model = Model(model_name="intfloat_multilingual_e5_large_instruct")
evaluate_model(
    model_path=model.local_path,
    index_name="uc202-rex-embeddings-e5",
    test_path="/opt/app-root/src/uc202-ipn-rex/notebooks/combined_annotations.xlsx",
    k_values=[3, 5, 10, 20, 30, 50, 100],
    only_2=False,
)


/tmp/ipykernel_45291/2001718618.py:20: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: list(zip(x["Content"], x["Annotation"])))
/tmp/ipykernel_45291/2001718618.py:50: DeprecationWarning: The 'body' parameter is deprecated and will be removed in a future version. Instead use individual parameters.
  response = es.search(index=index_name, body=query_body)
/opt/app-root/src/uc202-ipn-rex/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'noeyyalp.noe.edf.fr'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnin

Query ' Rejets d'effluents liquides non radioactifs.
Comment maitriser les déchets liquides composés d'eau de pluie qui a lessivé des zones de stockage matériel (ferrailles, sacs de ciment)? ' took 18.1205 seconds


/opt/app-root/src/uc202-ipn-rex/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'noeyyalp.noe.edf.fr'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Query 'A quelle vitesse sait-on mettre en oeuvre de grandes quantités de béton sur chantier ?' took 22.3178 seconds


/opt/app-root/src/uc202-ipn-rex/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'noeyyalp.noe.edf.fr'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


...

Query 'Anomalie loi décroissance débit' took 18.1886 seconds


/opt/app-root/src/uc202-ipn-rex/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'noeyyalp.noe.edf.fr'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Query 'Anomalie thermique combustible' took 22.2975 seconds


/opt/app-root/src/uc202-ipn-rex/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'noeyyalp.noe.edf.fr'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


KeyboardInterrupt: 

In [ ]:
evaluate_model(
    model_path="/opt/app-root/src/uc202-ipn-rex/notebooks/models/test",
    index_name="uc202-rex-embeddings-e5-trained",
    test_path="/opt/app-root/src/uc202-ipn-rex/notebooks/combined_annotations.xlsx",
    k_values=[3, 5, 10, 20, 30, 50, 100],
    only_2=False,
)
